# Grounded Research Assistant
### A mini "AI Gateway", built the way an Applied Scientist would in Wiley's AI Studio

**The problem AI Studio solves:** large language models are fluent but not always *right*. In
science, a confident wrong answer is dangerous. The job of an Applied Scientist is to make AI
answers **grounded in trusted, peer-reviewed content**, **cited**, and **honest about what they do
not know**.

**What this notebook builds:** a small, working system that does exactly that.

1. A **trusted-content layer** over real, open scholarly sources (OpenAlex + arXiv).
2. An **AI Gateway**: that content exposed as an **MCP server**, the same pattern Wiley uses with
   Anthropic to bring research into Claude.
3. A **grounded agent** built with **LangGraph** that retrieves, answers **with citations using
   Claude**, and then **verifies its own answer against the sources** (a responsible-AI step).
4. **LangSmith tracing**, so every step is auditable.

**How it maps to the role:**

| This notebook | Wiley AI Studio |
|---|---|
| OpenAlex + arXiv | Wiley's peer-reviewed journals |
| MCP gateway | Wiley AI Gateway + the Anthropic/MCP partnership |
| Claude (Anthropic) | Wiley's Anthropic partnership |
| Cite-every-claim + refuse-to-fabricate + verify | "Responsible AI", "expert-validated", "trusted content" |
| Grounding the model in real science | The Applied Scientist's core job |

Everything runs on **real** data and a **real** model. Nothing here is simulated.

## The pipeline at a glance

```
        question
           |
   [ retrieve ]  --> trusted content layer (OpenAlex peer-reviewed / arXiv)
           |               (also exposed as an MCP gateway, Step 3)
   [ synthesize ] --> Claude drafts an answer using ONLY those sources, with [n] citations
           |
   [ verify ]     --> Claude checks: is every claim supported by the sources? (responsible AI)
           |
   grounded, cited, checked answer   (+ full LangSmith trace)
```

If retrieval finds nothing, the assistant **refuses to answer** rather than inventing one. That
refusal is a feature, not a failure.

## Step 1 - Setup

Install the stack, then load keys. You need an **Anthropic key** (Claude does the reasoning).
LangSmith is optional (it records traces). The trusted-content APIs are free and need no key.

In [ ]:
# %pip installs into the running kernel. Skip if these already import in your environment.
%pip install -q langchain langgraph langchain-anthropic langchain-mcp-adapters mcp requests

In [ ]:
import os, sys, getpass, json, subprocess
print("Kernel Python:", sys.executable)

# --- Anthropic key: Claude is the model. Required. getpass hides it and does not save it. ---
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

# Pick a Claude model. Sonnet is a good speed/quality balance; swap if your account differs.
MODEL = "claude-sonnet-5"   # alternatives: "claude-opus-5" (best), "claude-haiku-4-5-20251001" (cheap)

# --- LangSmith tracing (optional). Responsible AI means every step is auditable. ---
_ls = getpass.getpass("LangSmith API key (optional, press Enter to skip): ")
if _ls.strip():
    os.environ["LANGSMITH_API_KEY"] = _ls.strip()
    os.environ["LANGCHAIN_API_KEY"] = _ls.strip()      # older env name, kept for compatibility
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "grounded-research-assistant"
    print("LangSmith tracing ON, runs will appear in your 'grounded-research-assistant' project.")
else:
    print("LangSmith tracing OFF (no key). Everything still runs.")

# Colab-only: give MCP's stdio transport a usable stderr (harmless elsewhere).
from mcp.client.stdio import stdio_client as _sc
getattr(_sc, "__wrapped__", _sc).__defaults__ = (subprocess.DEVNULL,)
print("Setup complete.")

## Step 2 - The trusted-content layer (real sources)

An Applied Scientist starts from the data, not the model. These two functions fetch **real**
scholarly records:

- **OpenAlex** - an open index of **peer-reviewed** works (our stand-in for Wiley's journals).
- **arXiv** - **preprints** (scholarly, but not yet peer-reviewed, we label them honestly).

Both are free and need no key. Each record carries a **DOI or link**, so every answer can be
traced back to a real paper.

In [14]:
import requests, re, xml.etree.ElementTree as ET
UA = {"User-Agent": "grounded-research/1.0 (mailto:khanibtisam38@gmail.com)"}

def _clean_query(q):
    # OpenAlex treats * and ? as WILDCARDS, so a natural question ("...cathodes?") returns a 400.
    # Strip punctuation down to plain search terms (letters, digits, spaces, hyphens).
    return re.sub(r"[^\w\s-]", " ", q).strip()

def _abstract_from_inverted(inv):
    # OpenAlex returns abstracts as an inverted index {word: [positions]}. Rebuild the text.
    if not inv:
        return ""
    pos = {}
    for word, idxs in inv.items():
        for i in idxs:
            pos[i] = word
    return " ".join(pos[i] for i in sorted(pos))[:2500]

def search_openalex(query, k=5):
    # Query OpenAlex for peer-reviewed works, asking only for the fields we need.
    params = {"search": _clean_query(query), "per-page": k,
              "select": "title,authorships,publication_year,doi,primary_location,abstract_inverted_index,cited_by_count"}
    r = requests.get("https://api.openalex.org/works", params=params, headers=UA, timeout=30)
    r.raise_for_status()                      # turn HTTP errors into exceptions we can see
    out = []
    for w in r.json().get("results", []):
        loc = (w.get("primary_location") or {}).get("source") or {}   # journal/venue info
        out.append({
            "title": w.get("title"),
            "authors": [a["author"]["display_name"] for a in w.get("authorships", [])[:3]],
            "year": w.get("publication_year"),
            "venue": loc.get("display_name"),
            "doi": w.get("doi"),
            "citations": w.get("cited_by_count"),
            "abstract": _abstract_from_inverted(w.get("abstract_inverted_index")),
            "source": "OpenAlex (peer-reviewed)"})
    return out

def search_arxiv(query, k=5):
    # Query arXiv (Atom XML) for preprints and parse with the standard library.
    params = {"search_query": "all:" + _clean_query(query), "max_results": k}
    r = requests.get("http://export.arxiv.org/api/query", params=params, headers=UA, timeout=30)
    r.raise_for_status()
    ns = {"a": "http://www.w3.org/2005/Atom"}                          # Atom XML namespace
    out = []
    for e in ET.fromstring(r.text).findall("a:entry", ns):
        out.append({
            "title": (e.findtext("a:title", "", ns) or "").strip(),
            "authors": [a.findtext("a:name", "", ns) for a in e.findall("a:author", ns)][:3],
            "year": (e.findtext("a:published", "", ns) or "")[:4],
            "venue": "arXiv (preprint)",
            "doi": e.findtext("a:id", "", ns),                         # the arXiv URL acts as the id
            "abstract": (e.findtext("a:summary", "", ns) or "").strip()[:2000],
            "source": "arXiv (preprint)"})
    return out

print("Trusted-content functions ready.")

Trusted-content functions ready.


In [15]:
# Quick live check on a real materials-science question (proves the sources work).
demo = search_openalex("lithium ion battery cathode degradation", k=3)
for i, p in enumerate(demo, 1):
    print(f"[{i}] {p['title']}  ({p['year']}, {p['venue']}) - {p['citations']} citations")
    print("    DOI:", p["doi"])

[1] 30 Years of Lithium‐Ion Batteries  (2018, Advanced Materials) - 6016 citations
    DOI: https://doi.org/10.1002/adma.201800561
[2] Direct regeneration of degraded lithium-ion battery cathodes with a multifunctional organic lithium salt  (2023, Nature Communications) - 496 citations
    DOI: https://doi.org/10.1038/s41467-023-36197-6
[3] Understanding the Degradation Mechanisms of LiNi<sub>0.5</sub>Co<sub>0.2</sub>Mn<sub>0.3</sub>O<sub>2</sub> Cathode Material in Lithium Ion Batteries  (2013, Advanced Energy Materials) - 1110 citations
    DOI: https://doi.org/10.1002/aenm.201300787


## Step 3 - The AI Gateway (an MCP server)

This is the Wiley/Anthropic pattern. We expose the trusted-content layer as a **single MCP
endpoint** so *any* AI client (Claude Desktop, an agent, an IDE) can reach it the same way. We
write the server to a file, then connect a client and call it over the protocol, exactly what
"bringing research into Claude via MCP" means under the hood.

In [ ]:
%%writefile gateway_server.py
# ============================================================================
# The "AI Gateway" as an MCP server.
# This mirrors Wiley's Anthropic partnership: ONE endpoint (an MCP server) that
# brings TRUSTED, peer-reviewed research into any AI client (Claude, an agent).
# The tools below hit real, open scholarly APIs so the whole thing is shareable.
# ============================================================================
import requests, re, xml.etree.ElementTree as ET
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Research-Gateway")
UA = {"User-Agent": "grounded-research/1.0 (mailto:khanibtisam38@gmail.com)"}


def _clean_query(q):
    # OpenAlex treats * and ? as wildcards; strip punctuation to plain search terms.
    return re.sub(r"[^\w\s-]", " ", q).strip()


def _abstract_from_inverted(inv):
    # OpenAlex stores abstracts as an inverted index {word: [positions]}. Rebuild the text.
    if not inv:
        return ""
    pos = {}
    for word, idxs in inv.items():
        for i in idxs:
            pos[i] = word
    return " ".join(pos[i] for i in sorted(pos))[:2500]


def _openalex(query, k=5):
    # OpenAlex = a free, open index of PEER-REVIEWED works (title, authors, DOI, abstract).
    params = {"search": _clean_query(query), "per-page": k,
              "select": "title,authorships,publication_year,doi,primary_location,abstract_inverted_index,cited_by_count"}
    r = requests.get("https://api.openalex.org/works", params=params, headers=UA, timeout=30)
    r.raise_for_status()
    out = []
    for w in r.json().get("results", []):
        loc = (w.get("primary_location") or {}).get("source") or {}
        out.append({
            "title": w.get("title"),
            "authors": [a["author"]["display_name"] for a in w.get("authorships", [])[:3]],
            "year": w.get("publication_year"),
            "venue": loc.get("display_name"),
            "doi": w.get("doi"),
            "citations": w.get("cited_by_count"),
            "abstract": _abstract_from_inverted(w.get("abstract_inverted_index")),
            "source": "OpenAlex (peer-reviewed)"})
    return out


def _arxiv(query, k=5):
    # arXiv = preprints (not yet peer-reviewed, but scholarly). Atom XML, parsed with stdlib.
    params = {"search_query": "all:" + _clean_query(query), "max_results": k}
    r = requests.get("http://export.arxiv.org/api/query", params=params, headers=UA, timeout=30)
    r.raise_for_status()
    ns = {"a": "http://www.w3.org/2005/Atom"}
    out = []
    for e in ET.fromstring(r.text).findall("a:entry", ns):
        out.append({
            "title": (e.findtext("a:title", "", ns) or "").strip(),
            "authors": [a.findtext("a:name", "", ns) for a in e.findall("a:author", ns)][:3],
            "year": (e.findtext("a:published", "", ns) or "")[:4],
            "venue": "arXiv (preprint)",
            "doi": e.findtext("a:id", "", ns),
            "abstract": (e.findtext("a:summary", "", ns) or "").strip()[:2000],
            "source": "arXiv (preprint)"})
    return out


@mcp.tool()
def search_literature(query: str, source: str = "openalex", k: int = 5) -> list:
    """Search trusted scholarly literature. source = "openalex" (peer-reviewed) or "arxiv" (preprints).
    Returns papers with title, authors, year, venue, DOI, and abstract, the trusted content an
    AI answer can be grounded in and cited to."""
    try:
        return _arxiv(query, k) if source == "arxiv" else _openalex(query, k)
    except Exception as exc:
        return [{"error": "literature search failed: " + str(exc)}]


if __name__ == "__main__":
    mcp.run()


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Launch the gateway as a subprocess and pass our environment through.
gateway = StdioServerParameters(command="python", args=["gateway_server.py"], env={**os.environ})

async def try_gateway():
    async with stdio_client(gateway) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()                         # MCP handshake
            tools = await session.list_tools()
            print("Gateway tools:", [t.name for t in tools.tools])
            # Call the tool over MCP and show it returns real trusted content.
            res = await session.call_tool("search_literature",
                                          {"query": "solid state electrolyte", "source": "openalex", "k": 3})
            papers = json.loads(res.content[0].text)
            for i, p in enumerate(papers, 1):
                print(f"[{i}] {p['title']} ({p.get('year')})")

await try_gateway()

We now have a working gateway. **Design note:** in a full product the agent below would reach
content *through* this MCP gateway (just as Claude reaches Wiley's content). To keep the agent
deterministic and easy to read, the next step calls the **same** trusted-content functions
directly. It is the identical content layer, one wrapped in MCP, one called in-process.

## Step 4 - The grounded agent (LangGraph + Claude)

Now the applied-science part. We build a **LangGraph** pipeline with three nodes:

1. **retrieve** - pull real papers for the question.
2. **synthesize** - Claude drafts an answer using **only** those papers, citing each claim `[n]`.
3. **verify** - Claude re-reads its own answer against the sources and flags anything unsupported.

The rules that make it *responsible* live in the system prompts: use only the given sources, cite
everything, and if the sources do not cover it, **say so** rather than inventing.

In [16]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatAnthropic(model=MODEL)   # newer Claude models manage sampling themselves (temperature is deprecated)

# The state is the shared "notepad" passed between nodes.
class State(TypedDict):
    question: str
    papers: List[dict]
    context: str
    answer: str
    grounding: str

# --- Node 1: retrieve real trusted content ---
def retrieve_node(state):
    papers = search_openalex(state["question"], k=8)   # peer-reviewed first; more sources = fuller answer
    if len(papers) < 4:                                 # thin? top up with preprints, labelled honestly
        papers += search_arxiv(state["question"], k=4)
    return {"papers": papers}

# --- Node 2: synthesize a grounded, cited answer with Claude ---
SYNTH_RULES = (
    "You are a scientific research assistant for professional users. "
    "Answer the question using ONLY the numbered sources provided. "
    "Cite every claim with its source number like [1], [2]. "
    "If the sources do not cover part of the question, say so plainly. "
    "Never invent facts, numbers, or citations. Be thorough and well-organised, "
    "drawing out the specific mechanisms and findings the sources actually contain.")

def synthesize_node(state):
    papers = state["papers"]
    if not papers:                                      # responsible AI: no sources -> no answer
        return {"answer": "I could not find supporting sources for this question, so I will not "
                          "answer it. (No fabrication.)", "context": ""}
    # Build a numbered context block the model must cite against.
    context = "\n\n".join(
        f"[{i}] {p['title']} ({p.get('year')}, {p.get('venue')}). {p.get('abstract','')[:1500]}"
        for i, p in enumerate(papers, 1))
    resp = llm.invoke([SystemMessage(content=SYNTH_RULES),
                       HumanMessage(content=f"Question: {state['question']}\n\nSources:\n{context}")])
    return {"answer": resp.content, "context": context}

# --- Node 3: verify the answer against the sources (the responsible-AI check) ---
VERIFY_RULES = (
    "You are a grounding checker. Given an ANSWER and its SOURCES, list any statements in the "
    "answer that are NOT supported by the sources. If every statement is supported, reply exactly: "
    "All claims grounded.")

def verify_node(state):
    if not state.get("context"):
        return {"grounding": "No sources were retrieved; the assistant correctly refused to answer."}
    resp = llm.invoke([SystemMessage(content=VERIFY_RULES),
                       HumanMessage(content=f"ANSWER:\n{state['answer']}\n\nSOURCES:\n{state['context']}")])
    return {"grounding": resp.content}

# --- Wire the nodes into a graph: retrieve -> synthesize -> verify ---
graph = StateGraph(State)
graph.add_node("retrieve", retrieve_node)
graph.add_node("synthesize", synthesize_node)
graph.add_node("verify", verify_node)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "synthesize")
graph.add_edge("synthesize", "verify")
graph.add_edge("verify", END)
assistant = graph.compile()
print("Grounded assistant compiled:", [n for n in assistant.get_graph().nodes])

Grounded assistant compiled: ['__start__', 'retrieve', 'synthesize', 'verify', '__end__']


## Step 5 - Ask it a real question

Watch the full behaviour: a cited answer, the list of real sources it used, and the grounding
check on its own output.

In [17]:
def ask(question):
    result = assistant.invoke({"question": question})
    print("QUESTION:", question, "\n")
    print("ANSWER (with citations):\n", result["answer"], "\n")
    print("GROUNDING CHECK:\n", result["grounding"], "\n")
    print("SOURCES USED:")
    for i, p in enumerate(result["papers"], 1):
        print(f"  [{i}] {p['title']} - {p.get('doi') or p.get('venue')}")
    return result

_ = ask("What are the main mechanisms that limit the cycle life of lithium-ion battery cathodes?")

QUESTION: What are the main mechanisms that limit the cycle life of lithium-ion battery cathodes? 

ANSWER (with citations):
 [{'signature': 'EowICokBCBAYAipAXl9DgTNpUajbCOcQ0uQzLzT3WNjz15Pi+3Oqv6QxfG5h/WSEE1l6MFX8+6aUAJYrg69j32z7qvliPSOQBzjXBDIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRhZTNlMmNjYy1mMTcyLTRhMGMtYjQ3ZC0wNjRkOTUyODgxMTQSDPIgrwjNClAGRsZJJRoMpsSjG46EvRIfCU40IjBQdtbAO7WjbeQ7zV3KI/IiKiE5a5+m6hbJrUjtk25R4Y07zdNFn3w19ZiltmC5wV4qrwbfNaAFMvIgt26ruT5qOJyKLhn3p3PWn55SRhQQo8fg0/y36MyFCj7fGpU5NUzN5aBXsz3zCy9uRqnGqt9j/VYkMJa1OFt49TMrqO9ibP7HOG+zwqKcNwTSYUsx6a/GBUhlkKS6nRAl1gTXP81MoOp2Ud+SroOrmWueKa2TsT74k75r2fOsZPA/19eNUfxzE24iOwoTDJXVH5o9peJjYxxSXnJKM3s9CAFllhpRdLljwPEYdiaJjC2hS/MH24uUu7nDi+geAHKN1R2QkLqLBZbTX+bgK7gJtuICWVQW9zjzv3BdmVzIsSi74xbYVU6kmZOXxAOfsTRh2FwoPfpx0fbIMWYDzUNBwMt5iWF3UB2FPtzMSL7CRelDP9IfyWB6u7BUhieZg9z5i7hxLFvdIF/1n3UCoMmWCTkwhg6Px0FrsvPBzQjJN2C44IV9ZtLiL78Ti7qI/O4t3ax9hWnDx9MIsqV/6FePFXI+YJQzUn3Vzh03SBVmP0onjMEBEas+kbQyPlHkXrZoNCjVkvQRXvYJKFma7CAI6IvVEZzx+IC+XSW2IL

## Step 6 - Responsible AI: watch it refuse

A grounded system must know its limits. Ask something no real paper will answer. Instead of
inventing a confident response, the assistant **declines**. In science, that honesty is the
product.

In [ ]:
_ = ask("What is the exact band gap of the fictional compound Unobtainium-7 zirconate?")

## Step 7 - Traceability (LangSmith)

If you entered a LangSmith key in Step 1, every run above was **traced**: each node, each Claude
call, each token and latency is recorded in your `grounded-research-assistant` project. That is
what "responsible AI" means in practice, not just good behaviour, but an **audit trail** you can
show a customer or a regulator. Open [smith.langchain.com](https://smith.langchain.com) to view it.

## Step 8 - What you built, and how to present it

You built a small but complete **AI Studio-style system**: trusted content, an MCP gateway,
Claude, an agentic LangGraph pipeline, grounding, citations, self-verification, and tracing.

**Say this in the interview:**
> "I built a working version of the pattern your team lives in: trusted, peer-reviewed research
> brought into Claude through an MCP gateway, with a responsible-AI layer that cites every claim,
> refuses to fabricate, and verifies its own answers, all traced in LangSmith. I used open
> sources so it is fully shareable, and demoed it on materials science."

**Honest limits (state them, it builds trust):**
- OpenAlex and arXiv stand in for Wiley's own peer-reviewed journals; the pattern is identical, the
  content source would change.
- Preprints are labelled as not-yet-peer-reviewed.
- The verifier is a strong first line, not a proof; a production system would add human review.

**Natural next steps:** a Gradio UI for non-experts, swapping the gateway onto a licensed journal
API, and adding a confidence/coverage score per answer.